# Named entity recognition with OpenVINO™


The Named Entity Recognition(NER) is a natural language processing method that involves the detecting of key information in the unstructured text and categorizing it into pre-defined categories. These categories or named entities refer to the key subjects of text, such as names, locations, companies and etc.

NER is a good method for the situations when a high-level overview of a large amount of text is needed. NER can be helpful with such task as analyzing key information in unstructured text or automates the information extraction of large amounts of data.


This tutorial shows how to perform named entity recognition using OpenVINO. We will use the pre-trained model [`elastic/distilbert-base-cased-finetuned-conll03-english`](https://huggingface.co/elastic/distilbert-base-cased-finetuned-conll03-english). It is DistilBERT based model, trained on `conll03 english dataset`. The model can recognize four named entities in text: persons, locations, organizations and names of miscellaneous entities that do not belong to the previous three groups. The model is sensitive to capital letters.

To simplify the user experience, the [Hugging Face Optimum](https://huggingface.co/docs/optimum) library is used to convert the model to OpenVINO™ IR format and quantize it.





### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/named-entity-recognition/named-entity-recognition.ipynb" />

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Download the NER model](#Download-the-NER-model)
- [Select inference device](#Select-inference-device)
- [Running OpenVINO model inference](#Running-OpenVINO-model-inference)
- [Running OpenVINO model inference via Optimum Intel](#Running-OpenVINO-model-inference-via-Optimum-Intel)
- [Quantize the model, using Hugging Face Optimum API](#Quantize-the-model,-using-Hugging-Face-Optimum-API)
- [Compare the Original and Quantized Models](#Compare-the-Original-and-Quantized-Models)
    - [Compare performance](#Compare-performance)
    - [Compare size of the models](#Compare-size-of-the-models)
- [Prepare demo for Named Entity Recognition OpenVINO Runtime](#Prepare-demo-for-Named-Entity-Recognition-OpenVINO-Runtime)



## Prerequisites
[back to top ⬆️](#Table-of-contents:)


In [1]:
import platform

%pip install -q "diffusers>=0.17.1" "openvino>=2025.1.0" "openvino-tokenizers"  "nncf>=2.16.0" "gradio>=4.19" "transformers>=4.36.0" "torch>=2.1" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -q "git+https://github.com/huggingface/optimum-intel.git"

if platform.system() == "Darwin":
    %pip install -q "numpy<2.0"

## Download the NER model
[back to top ⬆️](#Table-of-contents:)

We load the [`distilbert-base-cased-finetuned-conll03-english`](https://huggingface.co/elastic/distilbert-base-cased-finetuned-conll03-english) model from the [Hugging Face Hub](https://huggingface.co/models) with [Hugging Face Transformers library](https://huggingface.co/docs/transformers/index)and Optimum Intel with OpenVINO integration. For model conversion we will use `optimum-cli` tool.

In [2]:
from pathlib import Path

import requests

if not Path("notebook_utils.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
    )
    open("notebook_utils.py", "w").write(r.text)

if not Path("cmd_helper.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py",
    )
    open("cmd_helper.py", "w").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("named-entity-recognition.ipynb")

from cmd_helper import optimum_cli

original_ner_model_dir = Path("original_ner_model")

model_id = "elastic/distilbert-base-cased-finetuned-conll03-english"

if not original_ner_model_dir.exists():
    optimum_cli(model_id, original_ner_model_dir)

## Select inference device
[back to top ⬆️](#Table-of-contents:)

In [3]:
from notebook_utils import device_widget

device = device_widget(exclude="NPU")

device

Dropdown(description='Device:', index=1, options=('CPU', 'AUTO'), value='AUTO')

## Running OpenVINO model inference
[back to top ⬆️](#Table-of-contents:)


We can use converted model using native OpenVINO API without any dependency on PyTorch and Transformers code. For data preprocessing we will use OpenVINO Tokenizers.

[OpenVINO Tokenizers](https://github.com/openvinotoolkit/openvino_tokenizers) is an OpenVINO extension and a Python library designed to streamline tokenizer conversion for seamless integration into your projects. It supports Python and C++ environments and is compatible with all major platforms: Linux, Windows, and MacOS. You can find more examples OpenVINO Tokenizers usage in this [notebook](../openvino-tokenizers/openvino-tokenizers.ipynb) 



In [4]:
import openvino as ov
import openvino_tokenizers  # noqa: F401
import numpy as np

core = ov.Core()
ov_model = core.compile_model(original_ner_model_dir / "openvino_model.xml", device.value)
ov_tokenizer = core.compile_model(original_ner_model_dir / "openvino_tokenizer.xml")
ov_detokenizer = core.compile_model(original_ner_model_dir / "openvino_detokenizer.xml")

inputs = ov_tokenizer(["Hello, my name is Alex and I live in Paris"])
logits = ov_model({"input_ids": inputs["input_ids"], "attention_mask": inputs["attention_mask"]})[0]

predictions = np.argmax(logits, axis=2)
input_tokens = ov_detokenizer(inputs["input_ids"].reshape(-1, 1))[0]

id2label = {0: "O", 1: "B-PER", 2: "I-PER", 3: "B-ORG", 4: "I-ORG", 5: "B-LOC", 6: "I-LOC", 7: "B-MISC", 8: "I-MISC"}

for token, label in zip(input_tokens, predictions[0]):
    if label != 0:
        label_name = id2label[label]
        print(f"{token} - {label_name}")

/home/ea/work/py311/lib/python3.11/site-packages/openvino/runtime/__init__.py:10: DeprecationWarning: The `openvino.runtime` module is deprecated and will be removed in the 2026.0 release. Please replace `openvino.runtime` with `openvino`.
  warnings.warn(


Alex - B-PER
Paris - B-LOC


## Running OpenVINO model inference via Optimum Intel
[back to top ⬆️](#Table-of-contents:)


`OVModelForTokenClassification` is represent model class for Named Entity Recognition task in Optimum Intel. Model class initialization starts with calling `from_pretrained` method. Usage `OVModelForTokenClassification` allows to run model using similar API like original Transformers model.

Additionally, Optimum Intel model is compatible with HuggingFace [pipelines](https://huggingface.co/docs/transformers/en/main_classes/pipelines) API.

In [5]:
from transformers import AutoTokenizer
from optimum.intel import OVModelForTokenClassification
import torch

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = OVModelForTokenClassification.from_pretrained(original_ner_model_dir, device=device.value)

inputs = tokenizer("Hello, my name is Alex and I live in Paris", return_tensors="pt")

logits = model(**inputs).logits
tokens = tokenizer.batch_decode(inputs.input_ids.reshape(-1, 1))

predictions = torch.argmax(logits, dim=2)

for token, label in zip(input_tokens, predictions[0]):
    if label != 0:
        label_name = id2label[label.item()]
        print(f"{token} - {label_name}")

2025-04-15 12:51:22.761184: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-15 12:51:22.775868: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744707082.790169  958965 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744707082.794673  958965 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-15 12:51:22.812038: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

Alex - B-PER
Paris - B-LOC


## Quantize the model, using Hugging Face Optimum API
[back to top ⬆️](#Table-of-contents:)

Post-training static quantization introduces an additional calibration step where data is fed through the network in order to compute the activations quantization parameters. For quantization it will be used [Hugging Face Optimum Intel API](https://huggingface.co/docs/optimum/intel/index).

To handle the NNCF quantization process we use class [`OVQuantizer`](https://huggingface.co/docs/optimum/intel/reference_ov#optimum.intel.OVQuantizer). The quantization with Hugging Face Optimum Intel API contains the next steps:
* Model class initialization starts with calling `from_pretrained()` method.
* Next we create calibration dataset with `get_calibration_dataset()` to use for the post-training static quantization calibration step. 
* After we quantize a model and save the resulting model in the OpenVINO IR format to save_directory with `quantize()` method. 
* Then we load the quantized model. The Optimum Inference models are API compatible with Hugging Face Transformers models and we can just replace `AutoModelForXxx` class with the corresponding `OVModelForXxx` class. So we use `OVModelForTokenClassification` to load the model.

In [6]:
from functools import partial
from optimum.intel import OVQuantizer, OVConfig, OVQuantizationConfig


def preprocess_fn(data, tokenizer):
    examples = []
    for data_chunk in data["tokens"]:
        examples.append(" ".join(data_chunk))

    return tokenizer(examples, padding=True, truncation=True, max_length=128)


# The directory where the quantized model will be saved
quantized_ner_model_dir = Path("quantized_ner_model")

if not quantized_ner_model_dir.exists():
    quantizer = OVQuantizer.from_pretrained(model)
    calibration_dataset = quantizer.get_calibration_dataset(
        "conll2003",
        preprocess_function=partial(preprocess_fn, tokenizer=tokenizer),
        num_samples=100,
        dataset_split="train",
        preprocess_batch=True,
        trust_remote_code=True,
    )

    # Apply static quantization and save the resulting model in the OpenVINO IR format
    ov_config = OVConfig(quantization_config=OVQuantizationConfig(num_samples=len(calibration_dataset)))
    quantizer.quantize(
        calibration_dataset=calibration_dataset,
        save_directory=quantized_ner_model_dir,
        ov_config=ov_config,
    )

README.md:   0%|          | 0.00/12.3k [00:00<?, ?B/s]

conll2003.py:   0%|          | 0.00/9.57k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Output()

Output()

Output()

Output()

In [7]:
from notebook_utils import device_widget

device = device_widget()

device

Dropdown(description='Device:', index=1, options=('CPU', 'AUTO'), value='AUTO')

In [8]:
# Load the quantized model
optimized_model = OVModelForTokenClassification.from_pretrained(quantized_ner_model_dir, device=device.value)

## Compare the Original and Quantized Models
[back to top ⬆️](#Table-of-contents:)

Compare the original [`distilbert-base-cased-finetuned-conll03-english`](https://huggingface.co/elastic/distilbert-base-cased-finetuned-conll03-english) model with quantized and converted to OpenVINO IR format models to see the difference.

### Compare performance
[back to top ⬆️](#Table-of-contents:)

As the Optimum Inference models are API compatible with Hugging Face Transformers models, we can just use `pipleine()` from [Hugging Face Transformers API](https://huggingface.co/docs/transformers/index) for inference.

In [9]:
from transformers import pipeline
import torch
import platform

additional_args = {}

if platform.processor() == "arm":
    additional_args = {"device": torch.device("cpu")}

ner_pipeline_optimized = pipeline("token-classification", model=optimized_model, tokenizer=tokenizer, **additional_args)

ner_pipeline_original = pipeline("token-classification", model=model, tokenizer=tokenizer, **additional_args)

Device set to use cpu
Device set to use cpu


In [10]:
import time
import numpy as np


def calc_perf(ner_pipeline):
    inference_times = []

    for data in calibration_dataset:
        text = " ".join(data["tokens"])
        start = time.perf_counter()
        ner_pipeline(text)
        end = time.perf_counter()
        inference_times.append(end - start)

    return np.median(inference_times)


print(f"Median inference time of quantized model: {calc_perf(ner_pipeline_optimized)} ")

print(f"Median inference time of original model: {calc_perf(ner_pipeline_original)} ")

Median inference time of quantized model: 0.00587007449939847 
Median inference time of original model: 0.0049262153916060925 


### Compare size of the models
[back to top ⬆️](#Table-of-contents:)


In [11]:
from pathlib import Path

fp_model_file = Path(original_ner_model_dir) / "openvino_model.bin"
print(f"Size of original model in Bytes is {fp_model_file.stat().st_size}")
print(f'Size of quantized model in Bytes is {Path(quantized_ner_model_dir, "openvino_model.bin").stat().st_size}')

Size of original model in Bytes is 260795556
Size of quantized model in Bytes is 65802752


## Prepare demo for Named Entity Recognition OpenVINO Runtime
[back to top ⬆️](#Table-of-contents:)

Now, you can try NER model on own text. Put your sentence to input text box, click Submit button, the model label the recognized entities in the text.

In [ ]:
if not Path("gradio_helper.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/notebooks/magika-content-type-recognition/gradio_helper.py"
    )
    open("gradio_helper.py", "w").write(r.text)

from gradio_helper import make_demo

demo = make_demo(ner_pipeline_optimized)

try:
    demo.launch(debug=True)
except Exception:
    demo.launch(share=True, debug=True)
# if you are launching remotely, specify server_name and server_port
# demo.launch(server_name='your server name', server_port='server port in int')
# Read more in the docs: https://gradio.app/docs/